In [ ]:
import os
import time
import re
import sqlite3
import urllib.parse  
import requests
import urllib3

#
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


BANCO_DADOS = r"X:\KENEDY\banco_livros1.db"
PASTA_DESTINO = r"X:\KENEDY\Livros_Baixados"
DELAY_DOWNLOAD = 3

def higienizar_nome_arquivo_windows(nome):
    """Remove apenas caracteres proibidos pelo Windows para poder salvar o arquivo físico local."""
    if not nome:
        return "Arquivo_Sem_Nome"
    nome_limpo = re.sub(r'[\\/*?:"<>|]', ' ', nome)
    return " ".join(nome_limpo.split()).strip()

def obter_livros_minerados():
    if not os.path.exists(BANCO_DADOS):
        print(f"-- Erro: O banco de dados não foi localizado em: {BANCO_DADOS} --")
        return
    
    conexao = sqlite3.connect(BANCO_DADOS)
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT id, titulo, download FROM livros 
        WHERE titulo NOT LIKE 'Erro%' 
          AND titulo NOT LIKE 'Página%' 
          AND titulo != 'Título não encontrado'
    """)
    registros = cursor.fetchall()
    conexao.close()
    return registros

def executar_downloads_em_massa():
    os.makedirs(PASTA_DESTINO, exist_ok=True)
    livros_para_baixar = obter_livros_minerados()
    
    if not livros_para_baixar:
        print("-- Nenhum livre elegível foi encontrado no banco de dados para download. --")
        return
        
    print(f"--- MOTOR DE DOWNLOADS VIA LINK DIRETO CORRIGIDO ---")
    print(f"Total de registros prontos no banco: {len(livros_para_baixar)}")
    print(f"Destino dos arquivos: {PASTA_DESTINO}\n")
    
    for doc_id, download_original, nome_dow in livros_para_baixar:
        download_salvamento = higienizar_nome_arquivo_windows(download_original)
        nome_arquivo_final = f"{download_salvamento}.pdf"
        caminho_completo = os.path.join(PASTA_DESTINO, nome_arquivo_final)
        
        
        if os.path.exists(caminho_completo) and os.path.getsize(caminho_completo) > 10240:
            continue
            
        
        titulo_codificado_url = urllib.parse.quote(f"{nome_dow}", safe='()')
        
        
        
        url_direta_download = f"https://www.kufunda.net/{titulo_codificado_url}"
        url_pagina_referer = f"https://www.kufunda.net/pubview.php?id={doc_id}"
        
        headers_dinamicos = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:130.0) Gecko/20100101 Firefox/130.0',
            'Referer': url_pagina_referer,
            'Accept': '*/*',
            'Accept-Language': 'pt-BR,pt;q=0.8,en-US;q=0.5,en;q=0.3'
        }
        
        print(f"[ID {doc_id}] ----- Baixando: {nome_arquivo_final}")
        
        try:
            resposta = requests.get(url_direta_download, headers=headers_dinamicos, timeout=20, verify=False, stream=True)
            
            if resposta.status_code == 200:
                with open(caminho_completo, 'wb') as arquivo_pdf:
                    for pedaco_bytes in resposta.iter_content(chunk_size=8192):
                        if pedaco_bytes:
                            arquivo_pdf.write(pedaco_bytes)
                
                
                if os.path.getsize(caminho_completo) < 5120:
                    print(f"[ID {doc_id}]  Resposta inválida/pequena. Removendo arquivo...")
                    os.remove(caminho_completo)
                else:
                    print(f"[ID {doc_id}]  Download concluído com sucesso!")
            else:
                print(f"[ID {doc_id}]  Arquivo indisponível no servidor (Status {resposta.status_code}).")
                
        except Exception as e:
            print(f"[ID {doc_id}]  Falha técnica: {e}")
            if os.path.exists(caminho_completo):
                os.remove(caminho_completo)
                
        time.sleep(DELAY_DOWNLOAD)

    print("\n--- PROCESSO DE DOWNLOADS FINALIZADO ---")

if __name__ == "__main__":
    executar_downloads_em_massa()


--- MOTOR DE DOWNLOADS VIA LINK DIRETO CORRIGIDO ---
Total de registros prontos no banco: 880
Destino dos arquivos: X:\KENEDY\Livros_Baixados

[ID 42] ----- Baixando: doutrina crista.pdf
[ID 42]  Download concluído com sucesso!
[ID 44] ----- Baixando: o que é filosofia.pdf
[ID 44]  Download concluído com sucesso!
[ID 45] ----- Baixando: dicionário de filosofia.pdf
[ID 45]  Download concluído com sucesso!
[ID 46] ----- Baixando: dicionario filosofico.pdf
[ID 46]  Download concluído com sucesso!
[ID 47] ----- Baixando: historia da filosofia antiga.pdf
[ID 47]  Download concluído com sucesso!
[ID 48] ----- Baixando: assim falava zatustra.pdf
[ID 48]  Download concluído com sucesso!
[ID 49] ----- Baixando: alem do bem e do mal.pdf
[ID 49]  Download concluído com sucesso!
[ID 50] ----- Baixando: anticristo.pdf
[ID 50]  Download concluído com sucesso!
[ID 51] ----- Baixando: ecce homo.pdf
[ID 51]  Download concluído com sucesso!
[ID 52] ----- Baixando: sobre verdade e mentira.pdf
[ID 52]  Do